# Transfer Learning with Pretrained CNNs
### A step-by-step, plain-language walkthrough

This notebook is written the way you'd read through a script in a real project: everything runs top to bottom, in order, with no hidden functions to jump around to. Every concept is explained in plain language first, then built with real Keras code right below it, so you can see exactly what each line does and why it's there.

**What we're building, in one sentence:** instead of training a CNN from scratch (which needs millions of images and a lot of compute), we borrow a CNN that Google already trained on 1.4 million ImageNet images, and we teach it our own, much smaller task by reusing its "vision" and only training a small new "decision" layer on top.

---

## 1. Setup

We start by importing the tools we need:

- **TensorFlow / Keras** - the library that builds and trains the neural networks.
- **`keras.applications`** - a built-in library of famous, already-trained CNN architectures (MobileNetV2, EfficientNet, ResNet, etc.) that we can load with one line of code.
- **NumPy / Matplotlib** - for handling arrays of numbers and drawing pictures so we can *see* what's going on.

We also fix a **random seed**. Parts of this pipeline involve randomness - how data is shuffled, how images get randomly flipped or rotated during augmentation, how weights are initialized. Fixing the seed means "use the same randomness every time," so if you re-run this notebook you get the same results, instead of slightly different ones each time.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

> **Tip:** in Colab, go to **Runtime → Change runtime type → GPU** before running this notebook. Training even a small CNN head is much faster on a GPU.

## 2. The Core Idea: Transfer Learning

Imagine you want to build a system that recognizes 100 kinds of everyday objects, but you only have a few hundred photos of each - nowhere near enough to teach a neural network to understand images from zero.

Good news: someone else already solved a very similar, much bigger problem. **MobileNetV2** and **EfficientNet** are neural networks that Google trained on **ImageNet** - 1.4 million photos across 1000 categories. In the process of learning to tell those 1000 categories apart, the network had to learn general-purpose visual skills: recognizing edges, textures, colors, shapes, fur, wheels, eyes, and so on. Those skills are useful for almost *any* image task, not just the original 1000 categories.

**Transfer learning** means: reuse that already-learned visual understanding, and only train a small new piece on top for your own task. There are two stages to this, and we'll do both:

1. **Feature extraction** - freeze the pretrained network completely (don't let it change at all) and just train a small new classifier on top of the features it produces.
2. **Fine-tuning** - once that small classifier has learned something reasonable, carefully unlock the *last few* layers of the pretrained network too, and let them adjust slightly to our specific images, using very small, careful updates so we don't wreck what it already knows.

---

## 3. A Small Dataset to Learn With: CIFAR-10

Before we build the full 102-class Caltech101 pipeline (which takes longer to train), we'll first walk through every concept using **CIFAR-10** - 60,000 small 32×32 color photos across 10 everyday categories (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck). It's small and fast to experiment with, which makes it perfect for understanding each piece before we commit to a longer training run.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('Training images:', x_train.shape)
print('Test images:', x_test.shape)

# Look at a few sample images so we know what we're working with
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[y_train[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Loading a Frozen, Pretrained Feature Extractor

### What does "frozen" mean?
Every layer in a neural network has **weights** - numbers the network adjusts during training to get better at its task. "Freezing" a layer means locking those numbers so they *can't* change anymore, no matter what training happens afterward. We freeze the pretrained backbone because we don't want to damage the valuable ImageNet knowledge it already has - we just want to use it as-is.

### What does `include_top=False` mean?
MobileNetV2, as originally trained, ends with a final layer that outputs "this is one of these 1000 ImageNet categories." We don't want that - our categories are different. `include_top=False` cuts off that final decision layer and keeps only the earlier part of the network: the part that turns an image into a rich description of *what's in it*, not a final label.

### What comes out of the backbone?
With the top removed, MobileNetV2 turns a 224×224 image into a **7×7 grid of 1280-number descriptions** - think of it as checking 49 different spots across the image, and describing each spot with 1280 different "detector" readings (this spot has some fur-like texture, some round-edge, some green color, etc.).

### Why `GlobalAveragePooling2D`?
A 7×7 grid of readings is awkward to feed into a simple classifier. `GlobalAveragePooling2D` averages all 49 spots together for each of the 1280 detectors, collapsing everything down into **one list of 1280 numbers per image** - a compact "fingerprint" that summarizes the whole image.

In [ ]:
# Load MobileNetV2, pretrained on ImageNet, without its
# classification head, expecting 224x224x3 images
base_model = keras.applications.MobileNetV2(
    weights='imagenet',
    input_shape=(224, 224, 3),
    include_top=False,
)

# Freeze every weight in the backbone -- it will not be updated
# during training, it only produces features for us to use
base_model.trainable = False

print('Total layers in the backbone:', len(base_model.layers))
print('Trainable parameters:', base_model.count_params()
      if base_model.trainable else 0)
print('Total parameters (frozen):', base_model.count_params())

Now we wire the backbone into a small standalone model: image in, 1280-number feature vector out. We use the **Functional API** here (`keras.Input` + calling layers on tensors) rather than wrapping this in a `def build_...():` function, so every step is visible directly in the notebook, the way you'd read through a script in a real project.

One detail worth calling out: `base_model(inputs, training=False)`. MobileNetV2 contains **BatchNormalization** layers, which behave differently depending on whether the model thinks it's "training" or not - during training they normalize using the *current batch's* statistics, but during inference they should use the *fixed* statistics learned during the original ImageNet training. Passing `training=False` here forces that inference behavior, keeping the frozen backbone acting as a truly fixed, consistent feature extractor.

In [ ]:
feature_inputs = keras.Input(shape=(224, 224, 3))
features = base_model(feature_inputs, training=False)
pooled_features = keras.layers.GlobalAveragePooling2D()(features)

feature_extractor = keras.Model(feature_inputs, pooled_features,
                                 name='frozen_feature_extractor')
feature_extractor.summary()

Let's sanity-check it on a real batch of images. We resize a few CIFAR-10 images up to 224×224 (what MobileNetV2 expects) and pass them through.

In [ ]:
sample_images = tf.image.resize(x_train[:8].astype('float32') / 255.0,
                                 (224, 224))
sample_features = feature_extractor(sample_images, training=False)

print('Input batch shape:', sample_images.shape)
print('Output feature shape:', sample_features.shape)
print('(8 images, each turned into a 1280-number fingerprint)')

---

## 5. Adding a Trainable Classification Head

The 1280-number fingerprint the backbone produces isn't a prediction yet - it's just a description. We need a small, **trainable** piece on top that learns to turn that description into an actual decision ("this is a cat", "this is a truck", etc.).

### `Dense(128, activation='relu')`
A `Dense` layer with 128 units learns to combine the 1280 incoming numbers into 128 new numbers that are more useful specifically for *our* classes. Unlike the frozen backbone, this layer starts with random weights and actually learns during training.

**ReLU** (Rectified Linear Unit) is an activation function - after the layer combines numbers together, ReLU simply zeroes out any negative results and leaves positive ones untouched. Without an activation function like this, stacking layers would be mathematically equivalent to just one layer, no matter how many you stack - ReLU is what lets the network learn genuinely complex, non-straight-line patterns.

### `Dense(num_classes, activation='softmax')`
The final layer has one output number per class. **Softmax** turns those raw numbers into probabilities that all add up to 1 - e.g. "80% cat, 15% dog, 5% something else" - so we get a genuine, interpretable prediction.

In [ ]:
num_classes = 10  # CIFAR-10 has 10 categories

head_inputs = feature_extractor.input
x = feature_extractor.output
x = keras.layers.Dense(128, activation='relu')(x)
head_outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

classification_model = keras.Model(head_inputs, head_outputs,
                                    name='cifar10_classifier')
classification_model.summary()

---

## 6. Data Augmentation

### The problem: overfitting
If a model sees the exact same training images over and over across many epochs, it can start memorizing them instead of learning the general pattern of what makes a cat look like a cat. This is called **overfitting** - great performance on training data, poor performance on new, unseen images.

### The fix: random, harmless tweaks
Data augmentation randomly distorts each training image a little bit, differently, every time it's shown to the model. This way the model effectively never sees the exact same image twice, which forces it to learn the *real* underlying pattern rather than exact pixel arrangements.

- **`RandomFlip('horizontal')`** - randomly mirrors the image left-right. A cat is still a cat flipped horizontally.
- **`RandomRotation(0.15)`** - randomly rotates the image by up to 15% of a full turn (≈ ±54°), helping the model handle photos that aren't perfectly upright.
- **`RandomZoom(0.15)`** - randomly zooms in or out by up to 15%, helping the model recognize objects at different sizes.
- **`RandomContrast(0.1)`** - randomly adjusts contrast by up to 10%, helping the model handle different lighting.

We set `seed=42` on every layer so the "randomness" is reproducible - re-running this notebook applies the same sequence of random tweaks, which matters for fair comparisons between experiments.

Importantly, augmentation only makes sense **during training**. Keras augmentation layers automatically switch themselves off during evaluation/prediction - you don't want to randomly distort the images you're actually trying to classify correctly.

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal', seed=SEED),
    keras.layers.RandomRotation(0.15, seed=SEED),
    keras.layers.RandomZoom(0.15, seed=SEED),
    keras.layers.RandomContrast(0.1, seed=SEED),
], name='data_augmentation')

# Visualize what augmentation does to one image
example_image = tf.expand_dims(x_train[3].astype('float32') / 255.0, 0)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
axes[0].imshow(example_image[0])
axes[0].set_title('Original')
axes[0].axis('off')
for i in range(1, 5):
    augmented = data_augmentation(example_image, training=True)
    axes[i].imshow(tf.clip_by_value(augmented[0], 0, 1))
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

---

## 7. Phase 1: Training the Head (Backbone Still Frozen)

Now let's actually train the CIFAR-10 classifier. We rebuild the model from scratch, this time inserting a **resize** step (CIFAR-10 images are 32×32, MobileNetV2 needs 224×224) and the **augmentation** step, right before the frozen backbone.

### Choosing a loss function
`sparse_categorical_crossentropy` measures how wrong the model's probability predictions are, compared to the true class. We use the *sparse* version because our labels are plain integers (0-9), not one-hot encoded vectors - if they were one-hot encoded, we'd use plain `categorical_crossentropy` instead.

### Choosing an optimizer
`Adam` is the optimizer - the algorithm that decides how to adjust the trainable weights after each batch, based on the loss. Adam adapts its step size automatically per-parameter, which generally makes it a fast, reliable default choice compared to plain SGD.

### The learning rate
`1e-3` (0.001) is a normal-sized learning rate - reasonable here because only the small, randomly-initialized head is being trained; the frozen backbone is untouched, so there's no risk of damaging pretrained knowledge yet.

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = keras.layers.Resizing(224, 224)(inputs)
x = data_augmentation(x)
x = keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

cifar_model = keras.Model(inputs, outputs, name='cifar10_transfer_model')

cifar_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

cifar_model.summary()

**Note on preprocessing:** every pretrained model expects its input images normalized in a very specific way - matching however it was normalized *during its own original training*. MobileNetV2 expects pixel values scaled to the range [-1, 1], which is exactly what `keras.applications.mobilenet_v2.preprocess_input` does. Using the wrong normalization (like a generic `/255.0` alone) quietly hurts accuracy, since the pretrained weights were tuned around a specific input range.

In [ ]:
cifar_history_head = cifar_model.fit(
    x_train.astype('float32') / 255.0, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=2,
)

---

## 8. Phase 2: Fine-Tuning the Top of the Backbone

Phase 1 only trained the small head, while the backbone stayed completely frozen - that gets decent results, but the backbone's knowledge is still purely generic ImageNet knowledge; it never adjusted to our specific images.

### Why only unfreeze the *last* layers?
- **Early layers** (near the input) learn very generic visual building blocks: edges, colors, simple textures. These are useful for almost any image task, so there's no need to change them.
- **Later layers** (near the output) learn more specialized, high-level combinations of those building blocks. These benefit most from adapting to our specific dataset.

### Why use a much smaller learning rate now?
The backbone's weights already encode a huge amount of useful knowledge. Large updates could wreck that knowledge in just a few steps - a problem called **catastrophic forgetting**. Using a learning rate roughly 10-100x smaller than Phase 1 means the backbone adjusts gently, nudging itself toward our data instead of being knocked off course.

### Why keep BatchNorm layers frozen even here?
BatchNormalization layers track running statistics (average and spread of activations) from the original ImageNet training. Retraining those statistics on a smaller, different dataset can destabilize the network. So even among the "unfrozen" top layers, we deliberately keep any BatchNorm layers frozen.

In [ ]:
# Unfreeze the whole backbone first...
base_model.trainable = True

# ...then re-freeze everything except the last 20 layers
unfreeze_from = len(base_model.layers) - 20
for layer in base_model.layers[:unfreeze_from]:
    layer.trainable = False

# Among the unfrozen top layers, keep BatchNorm layers frozen
for layer in base_model.layers[unfreeze_from:]:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(l.trainable for l in base_model.layers)
print(f'{trainable_count} of {len(base_model.layers)} backbone layers are now trainable')

In [ ]:
# Re-compile is required any time trainable status changes,
# now with a much lower learning rate
cifar_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

cifar_history_finetune = cifar_model.fit(
    x_train.astype('float32') / 255.0, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=2,
)

In [ ]:
test_loss, test_accuracy = cifar_model.evaluate(
    x_test.astype('float32') / 255.0, y_test, verbose=0)
print(f'CIFAR-10 test accuracy after fine-tuning: {test_accuracy:.4f}')

---

## 9. Training Callbacks: Automating Good Habits

For the full Caltech101 run below, we'll use three callbacks - small helpers that watch training and react automatically:

- **`EarlyStopping`** - watches validation accuracy, and stops training if it hasn't improved for a few epochs in a row (`patience`). This avoids wasting time and avoids overfitting further once the model has stopped genuinely improving. `restore_best_weights=True` rolls the model back to its best epoch, not just whatever the last epoch happened to be.
- **`ReduceLROnPlateau`** - watches validation loss, and automatically shrinks the learning rate if progress stalls, so the model can take smaller, more careful steps as it gets close to a good solution.
- **`ModelCheckpoint`** - continuously saves the best-performing version of the model to disk during training, so we always keep the best snapshot even if later epochs happen to get worse.

---

## 10. The Full Pipeline: Caltech101 (102 Classes)

Now we put everything together for the actual target task: a classifier over the **Caltech101** dataset - 101 real object categories plus 1 background/clutter category, 102 classes total. Target: **≥ 85% validation accuracy**.

We swap MobileNetV2 for **EfficientNetB0** here - a somewhat more accurate (though slightly heavier) pretrained backbone, which gives extra headroom to reach the 85% target on a dataset with relatively few images per class.

> This section downloads the dataset via `tensorflow_datasets` the first time it runs, and trains for more epochs than the CIFAR-10 walkthrough above - expect this to take a while, especially without a GPU.

In [ ]:
!pip install -q tensorflow_datasets
import tensorflow_datasets as tfds

### Loading and preparing the dataset

We load Caltech101's train/test splits, then build a `tf.data` pipeline for each: resize every image to 224×224, apply augmentation (training split only), and apply EfficientNet's own `preprocess_input` (its expected normalization is different from MobileNetV2's, so we must use the matching function).

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 102

(train_raw, val_raw), ds_info = tfds.load(
    'caltech101',
    split=['train', 'test'],
    as_supervised=True,
    with_info=True,
)

print('Number of classes reported by the dataset:',
      ds_info.features['label'].num_classes)

In [ ]:
caltech_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal', seed=SEED),
    keras.layers.RandomRotation(0.15, seed=SEED),
    keras.layers.RandomZoom(0.15, seed=SEED),
    keras.layers.RandomContrast(0.1, seed=SEED),
], name='caltech_augmentation')

efficientnet_preprocess = keras.applications.efficientnet.preprocess_input


def preprocess_train(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = caltech_augmentation(image)
    image = efficientnet_preprocess(image)
    return image, label


def preprocess_eval(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = efficientnet_preprocess(image)
    return image, label

# NOTE: preprocess_train/preprocess_eval are simple, one-purpose
# transformations passed straight into tf.data's .map() -- this
# is the one place small helper functions are used, since
# tf.data pipelines are conventionally written this way even in
# production code, rather than inlined.

In [ ]:
train_ds = (
    train_raw
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_raw
    .map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print('Datasets ready.')

### Building the model: frozen EfficientNetB0 + classification head

In [ ]:
efficientnet_base = keras.applications.EfficientNetB0(
    weights='imagenet',
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    pooling='avg',
)
efficientnet_base.trainable = False

caltech_inputs = keras.Input(shape=IMG_SIZE + (3,))
x = efficientnet_base(caltech_inputs, training=False)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(128, activation='relu')(x)
caltech_outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

caltech_model = keras.Model(caltech_inputs, caltech_outputs,
                             name='caltech101_classifier')
caltech_model.summary()

**What is `Dropout(0.2)` doing here?** During training, it randomly "turns off" 20% of the incoming numbers on each pass, forcing the head to not over-rely on any single feature. It's another tool (alongside data augmentation) for reducing overfitting, particularly useful here since Caltech101 has far fewer images per class than ImageNet did.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6,
    ),
    keras.callbacks.ModelCheckpoint(
        'caltech101_model.h5', monitor='val_accuracy',
        save_best_only=True,
    ),
]

### Phase 1: train the head with the backbone frozen

In [ ]:
caltech_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_head = caltech_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=2,
)

### Phase 2: unfreeze the top of EfficientNetB0 and fine-tune

Same reasoning as the CIFAR-10 walkthrough above: unlock the last 40 layers, keep BatchNorm frozen among them, and continue training at a much lower learning rate.

In [ ]:
efficientnet_base.trainable = True

unfreeze_from = len(efficientnet_base.layers) - 40
for layer in efficientnet_base.layers[:unfreeze_from]:
    layer.trainable = False
for layer in efficientnet_base.layers[unfreeze_from:]:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

caltech_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_finetune = caltech_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=2,
)

### Evaluating and saving the final model

In [ ]:
val_loss, val_accuracy = caltech_model.evaluate(val_ds, verbose=0)
print(f'Final validation accuracy: {val_accuracy:.4f}')
print(f'Target was >= 0.85 -- {"PASSED" if val_accuracy >= 0.85 else "NOT YET MET"}')

caltech_model.save('caltech101_model.h5')
print('Model saved to caltech101_model.h5')

In [ ]:
# Plot the full training history (both phases) so we can see
# how accuracy and loss evolved across both training stages
acc = history_head.history['accuracy'] + history_finetune.history['accuracy']
val_acc = (history_head.history['val_accuracy']
           + history_finetune.history['val_accuracy'])
loss = history_head.history['loss'] + history_finetune.history['loss']
val_loss_curve = (history_head.history['val_loss']
                  + history_finetune.history['val_loss'])
phase_boundary = len(history_head.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(acc, label='train accuracy')
ax1.plot(val_acc, label='val accuracy')
ax1.axvline(phase_boundary - 0.5, color='gray', linestyle='--',
            label='fine-tuning starts')
ax1.axhline(0.85, color='red', linestyle=':', label='85% target')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(loss, label='train loss')
ax2.plot(val_loss_curve, label='val loss')
ax2.axvline(phase_boundary - 0.5, color='gray', linestyle='--',
            label='fine-tuning starts')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

---

## 11. Loading the Saved Model Later

Once saved, the model (architecture + trained weights) can be reloaded at any time, without needing any of the code above, for direct inference on new images.

In [ ]:
reloaded_model = keras.models.load_model('caltech101_model.h5')

class_label_names = ds_info.features['label'].names

# Grab one batch from the validation set and predict on it
for images, labels in val_ds.take(1):
    predictions = reloaded_model.predict(images, verbose=0)
    predicted_classes = tf.argmax(predictions, axis=1)
    for i in range(5):
        true_name = class_label_names[labels[i].numpy()]
        pred_name = class_label_names[predicted_classes[i].numpy()]
        confidence = predictions[i][predicted_classes[i]] * 100
        print(f'True: {true_name:20s}  Predicted: {pred_name:20s} '
              f'({confidence:.1f}% confident)')

---
## 12. Glossary - Every Term Used Above

| Term | Plain-language meaning |
|---|---|
| **Transfer learning** | Reusing a model trained on one large task (ImageNet) to help solve a different, smaller task. |
| **ImageNet** | A dataset of 1.4 million labeled photos across 1000 categories, the standard benchmark pretrained CNNs are trained on. |
| **Backbone / base model** | The pretrained CNN whose convolutional layers extract visual features from an image. |
| **`include_top=False`** | Loads the pretrained CNN without its original final classification layer. |
| **Freezing a layer** | Locking its weights so they don't change during training. |
| **Feature vector** | A list of numbers summarizing what a network "sees" in an image. |
| **`GlobalAveragePooling2D`** | Averages a spatial grid of features down into a single vector, with no extra learned parameters. |
| **`Dense` layer** | A fully-connected layer that learns to combine its inputs into new, more useful numbers. |
| **ReLU** | An activation function that zeroes out negative values; lets networks learn non-linear patterns. |
| **Softmax** | Turns raw output numbers into probabilities that sum to 1. |
| **Classification head** | The small, trainable layers added on top of a frozen backbone to produce predictions. |
| **`BatchNormalization`** | A layer that stabilizes training by normalizing activations; behaves differently in training vs. inference mode. |
| **`training=False`** | Forces layers like BatchNorm to behave in inference mode, even inside a model that may later be trained. |
| **Fine-tuning** | Unfreezing some of a pretrained backbone's layers and training them further, gently, on new data. |
| **Catastrophic forgetting** | The risk of a model losing its useful pretrained knowledge if fine-tuned too aggressively. |
| **Learning rate** | How big a step the optimizer takes when updating weights; smaller during fine-tuning to avoid damage. |
| **Optimizer (Adam)** | The algorithm that decides how to adjust weights based on the loss, each training step. |
| **Loss function** | A number measuring how wrong the model's predictions are; the thing training tries to minimize. |
| **`sparse_categorical_crossentropy`** | A loss function for multi-class classification when labels are plain integers (not one-hot encoded). |
| **Data augmentation** | Randomly, harmlessly distorting training images (flip/rotate/zoom/contrast) to reduce overfitting. |
| **Overfitting** | When a model memorizes training data instead of learning general patterns, hurting performance on new data. |
| **`preprocess_input`** | The specific pixel-value normalization a given pretrained model expects, matching how it was originally trained. |
| **Dropout** | Randomly disables a fraction of neurons during training to reduce overfitting. |
| **Epoch** | One full pass through the entire training dataset. |
| **Batch size** | How many images are processed together before the model's weights are updated once. |
| **`verbose=2`** | Training log setting that prints one line per epoch (rather than an animated progress bar), useful for notebooks and logs. |
| **`EarlyStopping`** | Callback that halts training once validation performance stops improving. |
| **`ReduceLROnPlateau`** | Callback that shrinks the learning rate automatically when progress stalls. |
| **`ModelCheckpoint`** | Callback that saves the best-performing model seen so far during training. |
